In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 10: Automated Multi-Format Sales Report Pipeline and Delivery\n",
    "\n",
    "**Objective:** Design and implement a production-ready automated report pipeline that extracts, transforms, and generates sales reports in multiple formats (Excel, HTML, PDF) and optionally delivers them via email.\n",
    "\n",
    "**Reference:** Note 6.6 – Code Implementation"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 0: Import Required Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "All libraries imported successfully!\n",
      "pandas version  : 3.0.0\n",
      "openpyxl version: 3.1.5\n"
     ]
    }
   ],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import os\n",
    "import smtplib\n",
    "import matplotlib\n",
    "matplotlib.use('Agg') \n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "\n",
    "from datetime import datetime\n",
    "from email.mime.multipart import MIMEMultipart\n",
    "from email.mime.base import MIMEBase\n",
    "from email.mime.text import MIMEText\n",
    "from email import encoders\n",
    "\n",
    "# Excel\n",
    "import openpyxl\n",
    "from openpyxl.styles import PatternFill, Font, Alignment, Border, Side\n",
    "from openpyxl.utils import get_column_letter\n",
    "\n",
    "# Jinja2 for HTML\n",
    "from jinja2 import Template\n",
    "\n",
    "# FPDF2 for PDF\n",
    "from fpdf import FPDF\n",
    "\n",
    "print(\"All libraries imported successfully!\")\n",
    "print(f\"pandas version  : {pd.__version__}\")\n",
    "print(f\"openpyxl version: {openpyxl.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 1: EXTRACT – Load Sales Data\n",
    "\n",
    "In the **Extract** phase, raw data is loaded from its source. The goal is simply to bring the data into memory with no modifications yet."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Raw dataset shape: (120, 5)\n",
      "Columns: ['Month', 'Product', 'Units_Sold', 'Unit_Price', 'Region']\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Month</th>\n",
       "      <th>Product</th>\n",
       "      <th>Units_Sold</th>\n",
       "      <th>Unit_Price</th>\n",
       "      <th>Region</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>43</td>\n",
       "      <td>55000</td>\n",
       "      <td>North</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>33</td>\n",
       "      <td>55000</td>\n",
       "      <td>South</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>19</td>\n",
       "      <td>55000</td>\n",
       "      <td>East</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>47</td>\n",
       "      <td>55000</td>\n",
       "      <td>West</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>12</td>\n",
       "      <td>35000</td>\n",
       "      <td>North</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>25</td>\n",
       "      <td>35000</td>\n",
       "      <td>South</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>43</td>\n",
       "      <td>35000</td>\n",
       "      <td>East</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>23</td>\n",
       "      <td>35000</td>\n",
       "      <td>West</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Headphones</td>\n",
       "      <td>27</td>\n",
       "      <td>3000</td>\n",
       "      <td>North</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Headphones</td>\n",
       "      <td>15</td>\n",
       "      <td>3000</td>\n",
       "      <td>South</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "  Month     Product  Units_Sold  Unit_Price Region\n",
       "0   Jan      Laptop          43       55000  North\n",
       "1   Jan      Laptop          33       55000  South\n",
       "2   Jan      Laptop          19       55000   East\n",
       "3   Jan      Laptop          47       55000   West\n",
       "4   Jan      Tablet          12       35000  North\n",
       "5   Jan      Tablet          25       35000  South\n",
       "6   Jan      Tablet          43       35000   East\n",
       "7   Jan      Tablet          23       35000   West\n",
       "8   Jan  Headphones          27        3000  North\n",
       "9   Jan  Headphones          15        3000  South"
      ]
     },
     "execution_count": 2,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "np.random.seed(42)\n",
    "\n",
    "months   = [\"Jan\", \"Feb\", \"Mar\", \"Apr\", \"May\", \"Jun\"]\n",
    "products = [\"Laptop\", \"Tablet\", \"Headphones\", \"Monitor\", \"Keyboard\"]\n",
    "regions  = [\"North\", \"South\", \"East\", \"West\"]\n",
    "\n",
    "# Fixed unit prices per product\n",
    "unit_prices = {\n",
    "    \"Laptop\"     : 55000,\n",
    "    \"Tablet\"     : 35000,\n",
    "    \"Headphones\" :  3000,\n",
    "    \"Monitor\"    : 15000,\n",
    "    \"Keyboard\"   :  2000,\n",
    "}\n",
    "\n",
    "rows = []\n",
    "for month in months:\n",
    "    for product in products:\n",
    "        for region in regions:\n",
    "            units = np.random.randint(5, 50)\n",
    "            rows.append({\n",
    "                \"Month\"      : month,\n",
    "                \"Product\"    : product,\n",
    "                \"Units_Sold\" : units,\n",
    "                \"Unit_Price\" : unit_prices[product],\n",
    "                \"Region\"     : region,\n",
    "            })\n",
    "\n",
    "df_raw = pd.DataFrame(rows)\n",
    "\n",
    "print(f\"Raw dataset shape: {df_raw.shape}\")\n",
    "print(f\"Columns: {list(df_raw.columns)}\")\n",
    "df_raw.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 2: TRANSFORM – Clean and Enrich Data\n",
    "\n",
    "In the **Transform** phase we:\n",
    "- Calculate `Revenue = Units_Sold × Unit_Price`\n",
    "- Aggregate to product-level summaries (Total Units, Total Revenue, Average Price, Revenue Share %)\n",
    "- Aggregate to monthly totals for charting"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Sample rows after Revenue calculation:\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Month</th>\n",
       "      <th>Product</th>\n",
       "      <th>Units_Sold</th>\n",
       "      <th>Unit_Price</th>\n",
       "      <th>Region</th>\n",
       "      <th>Revenue</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>43</td>\n",
       "      <td>55000</td>\n",
       "      <td>North</td>\n",
       "      <td>2365000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>33</td>\n",
       "      <td>55000</td>\n",
       "      <td>South</td>\n",
       "      <td>1815000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>19</td>\n",
       "      <td>55000</td>\n",
       "      <td>East</td>\n",
       "      <td>1045000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Laptop</td>\n",
       "      <td>47</td>\n",
       "      <td>55000</td>\n",
       "      <td>West</td>\n",
       "      <td>2585000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>12</td>\n",
       "      <td>35000</td>\n",
       "      <td>North</td>\n",
       "      <td>420000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>Jan</td>\n",
       "      <td>Tablet</td>\n",
       "      <td>25</td>\n",
       "      <td>35000</td>\n",
       "      <td>South</td>\n",
       "      <td>875000</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "  Month Product  Units_Sold  Unit_Price Region  Revenue\n",
       "0   Jan  Laptop          43       55000  North  2365000\n",
       "1   Jan  Laptop          33       55000  South  1815000\n",
       "2   Jan  Laptop          19       55000   East  1045000\n",
       "3   Jan  Laptop          47       55000   West  2585000\n",
       "4   Jan  Tablet          12       35000  North   420000\n",
       "5   Jan  Tablet          25       35000  South   875000"
      ]
     },
     "execution_count": 3,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# ── 2a. Calculate Revenue ──────────────────────────────────────────────────\n",
    "df_raw[\"Revenue\"] = df_raw[\"Units_Sold\"] * df_raw[\"Unit_Price\"]\n",
    "\n",
    "print(\"Sample rows after Revenue calculation:\")\n",
    "df_raw.head(6)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Product Summary:\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Product</th>\n",
       "      <th>Total_Units</th>\n",
       "      <th>Total_Revenue</th>\n",
       "      <th>Avg_Price</th>\n",
       "      <th>Revenue_Share_Pct</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Laptop</td>\n",
       "      <td>684</td>\n",
       "      <td>37620000</td>\n",
       "      <td>55000.0</td>\n",
       "      <td>50.17</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Tablet</td>\n",
       "      <td>724</td>\n",
       "      <td>25340000</td>\n",
       "      <td>35000.0</td>\n",
       "      <td>33.80</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Monitor</td>\n",
       "      <td>591</td>\n",
       "      <td>8865000</td>\n",
       "      <td>15000.0</td>\n",
       "      <td>11.82</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Headphones</td>\n",
       "      <td>607</td>\n",
       "      <td>1821000</td>\n",
       "      <td>3000.0</td>\n",
       "      <td>2.43</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>Keyboard</td>\n",
       "      <td>666</td>\n",
       "      <td>1332000</td>\n",
       "      <td>2000.0</td>\n",
       "      <td>1.78</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "      Product  Total_Units  Total_Revenue  Avg_Price  Revenue_Share_Pct\n",
       "0      Laptop          684       37620000    55000.0              50.17\n",
       "1      Tablet          724       25340000    35000.0              33.80\n",
       "2     Monitor          591        8865000    15000.0              11.82\n",
       "3  Headphones          607        1821000     3000.0               2.43\n",
       "4    Keyboard          666        1332000     2000.0               1.78"
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# ── 2b. Product-level summary ──────────────────────────────────────────────\n",
    "product_summary = (\n",
    "    df_raw\n",
    "    .groupby(\"Product\", as_index=False)\n",
    "    .agg(\n",
    "        Total_Units   = (\"Units_Sold\", \"sum\"),\n",
    "        Total_Revenue = (\"Revenue\",    \"sum\"),\n",
    "        Avg_Price     = (\"Unit_Price\", \"mean\"),\n",
    "    )\n",
    ")\n",
    "\n",
    "total_rev = product_summary[\"Total_Revenue\"].sum()\n",
    "product_summary[\"Revenue_Share_Pct\"] = (\n",
    "    product_summary[\"Total_Revenue\"] / total_rev * 100\n",
    ").round(2)\n",
    "\n",
    "product_summary = product_summary.sort_values(\"Total_Revenue\", ascending=False).reset_index(drop=True)\n",
    "\n",
    "print(\"Product Summary:\")\n",
    "product_summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Monthly Revenue:\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Month</th>\n",
       "      <th>Monthly_Revenue</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Jan</td>\n",
       "      <td>13671000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Feb</td>\n",
       "      <td>12334000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Mar</td>\n",
       "      <td>9846000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Apr</td>\n",
       "      <td>11902000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>May</td>\n",
       "      <td>12949000</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>Jun</td>\n",
       "      <td>14276000</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "  Month  Monthly_Revenue\n",
       "0   Jan         13671000\n",
       "1   Feb         12334000\n",
       "2   Mar          9846000\n",
       "3   Apr         11902000\n",
       "4   May         12949000\n",
       "5   Jun         14276000"
      ]
     },
     "execution_count": 5,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# ── 2c. Monthly total revenue ──────────────────────────────────────────────\n",
    "month_order = [\"Jan\", \"Feb\", \"Mar\", \"Apr\", \"May\", \"Jun\"]\n",
    "\n",
    "monthly_revenue = (\n",
    "    df_raw\n",
    "    .groupby(\"Month\", as_index=False)\n",
    "    .agg(Monthly_Revenue = (\"Revenue\", \"sum\"))\n",
    ")\n",
    "monthly_revenue[\"Month\"] = pd.Categorical(\n",
    "    monthly_revenue[\"Month\"], categories=month_order, ordered=True\n",
    ")\n",
    "monthly_revenue = monthly_revenue.sort_values(\"Month\").reset_index(drop=True)\n",
    "\n",
    "print(\"Monthly Revenue:\")\n",
    "monthly_revenue"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 3: Prepare the `reports/` Output Folder"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Reports will be saved to: ./reports/\n",
      "  Excel : reports\\sales_report_20260312.xlsx\n",
      "  HTML  : reports\\sales_report_20260312.html\n",
      "  PDF   : reports\\sales_report_20260312.pdf\n"
     ]
    }
   ],
   "source": [
    "# Create reports directory if it does not exist\n",
    "REPORTS_DIR = \"reports\"\n",
    "os.makedirs(REPORTS_DIR, exist_ok=True)\n",
    "\n",
    "# Date string for filenames\n",
    "date_str = datetime.now().strftime(\"%Y%m%d\")\n",
    "\n",
    "EXCEL_PATH = os.path.join(REPORTS_DIR, f\"sales_report_{date_str}.xlsx\")\n",
    "HTML_PATH  = os.path.join(REPORTS_DIR, f\"sales_report_{date_str}.html\")\n",
    "PDF_PATH   = os.path.join(REPORTS_DIR, f\"sales_report_{date_str}.pdf\")\n",
    "CHART_PATH = os.path.join(REPORTS_DIR, f\"monthly_revenue_chart_{date_str}.png\")\n",
    "\n",
    "print(f\"Reports will be saved to: ./{REPORTS_DIR}/\")\n",
    "print(f\"  Excel : {EXCEL_PATH}\")\n",
    "print(f\"  HTML  : {HTML_PATH}\")\n",
    "print(f\"  PDF   : {PDF_PATH}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 4: GENERATE – Excel Report (openpyxl)\n",
    "\n",
    "We generate an Excel workbook with two sheets:\n",
    "- **Product Summary** – styled headers, formatted currency columns\n",
    "- **Monthly Revenue** – data used for the bar chart"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[Excel] Saved → reports\\sales_report_20260312.xlsx\n"
     ]
    }
   ],
   "source": [
    "def generate_excel(product_summary: pd.DataFrame,\n",
    "                   monthly_revenue: pd.DataFrame,\n",
    "                   path: str) -> None:\n",
    "    \"\"\"Generate a styled Excel report with two sheets.\"\"\"\n",
    "\n",
    "    wb = openpyxl.Workbook()\n",
    "\n",
    "    # ── Helper styles ──────────────────────────────────────────────────────\n",
    "    header_fill  = PatternFill(\"solid\", fgColor=\"1F4E79\")   \n",
    "    alt_fill     = PatternFill(\"solid\", fgColor=\"D6E4F0\")   \n",
    "    header_font  = Font(color=\"FFFFFF\", bold=True, size=11)\n",
    "    body_font    = Font(size=11)\n",
    "    center_align = Alignment(horizontal=\"center\", vertical=\"center\")\n",
    "    thin = Side(style=\"thin\", color=\"BFBFBF\")\n",
    "    border = Border(left=thin, right=thin, top=thin, bottom=thin)\n",
    "\n",
    "    # ── Sheet 1: Product Summary ───────────────────────────────────────────\n",
    "    ws1 = wb.active\n",
    "    ws1.title = \"Product Summary\"\n",
    "\n",
    "    headers = [\"Product\", \"Total Units\", \"Total Revenue (Rs)\",\n",
    "               \"Avg Unit Price (Rs)\", \"Revenue Share (%)\"]\n",
    "    ws1.append(headers)\n",
    "\n",
    "    # Style header row\n",
    "    for col_idx, _ in enumerate(headers, start=1):\n",
    "        cell = ws1.cell(row=1, column=col_idx)\n",
    "        cell.fill      = header_fill\n",
    "        cell.font      = header_font\n",
    "        cell.alignment = center_align\n",
    "        cell.border    = border\n",
    "\n",
    "    # Data rows\n",
    "    for row_idx, row in product_summary.iterrows():\n",
    "        data_row = [\n",
    "            row[\"Product\"],\n",
    "            int(row[\"Total_Units\"]),\n",
    "            float(row[\"Total_Revenue\"]),\n",
    "            float(row[\"Avg_Price\"]),\n",
    "            float(row[\"Revenue_Share_Pct\"]),\n",
    "        ]\n",
    "        ws1.append(data_row)\n",
    "        excel_row = row_idx + 2   # +1 for header, +1 for 1-indexing\n",
    "        fill = alt_fill if row_idx % 2 == 0 else PatternFill()\n",
    "        for col_idx in range(1, len(headers) + 1):\n",
    "            cell = ws1.cell(row=excel_row, column=col_idx)\n",
    "            cell.fill      = fill\n",
    "            cell.font      = body_font\n",
    "            cell.alignment = center_align\n",
    "            cell.border    = border\n",
    "            # Format currency columns\n",
    "            if col_idx in (3, 4):\n",
    "                cell.number_format = '#,##0.00'\n",
    "            elif col_idx == 5:\n",
    "                cell.number_format = '0.00\"%\"'\n",
    "\n",
    "    # Auto-fit column widths\n",
    "    col_widths = [18, 14, 22, 22, 18]\n",
    "    for i, width in enumerate(col_widths, start=1):\n",
    "        ws1.column_dimensions[get_column_letter(i)].width = width\n",
    "\n",
    "    # ── Sheet 2: Monthly Revenue ───────────────────────────────────────────\n",
    "    ws2 = wb.create_sheet(title=\"Monthly Revenue\")\n",
    "    ws2.append([\"Month\", \"Total Revenue (Rs)\"])\n",
    "    for col_idx in range(1, 3):\n",
    "        cell = ws2.cell(row=1, column=col_idx)\n",
    "        cell.fill = header_fill\n",
    "        cell.font = header_font\n",
    "        cell.alignment = center_align\n",
    "\n",
    "    for _, row in monthly_revenue.iterrows():\n",
    "        ws2.append([str(row[\"Month\"]), float(row[\"Monthly_Revenue\"])])\n",
    "\n",
    "    ws2.column_dimensions[\"A\"].width = 12\n",
    "    ws2.column_dimensions[\"B\"].width = 22\n",
    "\n",
    "    wb.save(path)\n",
    "    print(f\"[Excel] Saved → {path}\")\n",
    "\n",
    "\n",
    "generate_excel(product_summary, monthly_revenue, EXCEL_PATH)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 5: GENERATE – HTML Report (Jinja2)\n",
    "\n",
    "We use a **Jinja2 template** to render a fully styled HTML report that includes the product summary table and the total revenue figure."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[HTML] Saved → reports\\sales_report_20260312.html\n"
     ]
    }
   ],
   "source": [
    "HTML_TEMPLATE = \"\"\"\n",
    "<!DOCTYPE html>\n",
    "<html lang=\"en\">\n",
    "<head>\n",
    "  <meta charset=\"UTF-8\">\n",
    "  <title>Sales Report – {{ report_date }}</title>\n",
    "  <style>\n",
    "    body  { font-family: Arial, sans-serif; margin: 40px; color: #333; }\n",
    "    h1    { color: #1F4E79; border-bottom: 3px solid #1F4E79; padding-bottom: 8px; }\n",
    "    h2    { color: #2E75B6; margin-top: 30px; }\n",
    "    .meta { color: #666; font-size: 14px; margin-bottom: 20px; }\n",
    "    table { border-collapse: collapse; width: 100%; margin-top: 12px; }\n",
    "    th    { background: #1F4E79; color: #fff; padding: 10px 14px; text-align: center; }\n",
    "    td    { padding: 9px 14px; text-align: center; border: 1px solid #ddd; }\n",
    "    tr:nth-child(even) td { background: #D6E4F0; }\n",
    "    tr:hover td { background: #BDD7EE; }\n",
    "    .total-box {\n",
    "      margin-top: 20px; padding: 14px 20px;\n",
    "      background: #1F4E79; color: #fff;\n",
    "      display: inline-block; border-radius: 6px; font-size: 16px;\n",
    "    }\n",
    "    footer { margin-top: 40px; font-size: 12px; color: #aaa; }\n",
    "  </style>\n",
    "</head>\n",
    "<body>\n",
    "  <h1>📊 Automated Sales Report</h1>\n",
    "  <p class=\"meta\">Generated on: <strong>{{ report_date }}</strong></p>\n",
    "\n",
    "  <h2>Product Summary</h2>\n",
    "  <table>\n",
    "    <thead>\n",
    "      <tr>\n",
    "        <th>#</th>\n",
    "        <th>Product</th>\n",
    "        <th>Total Units Sold</th>\n",
    "        <th>Total Revenue (Rs)</th>\n",
    "        <th>Avg Unit Price (Rs)</th>\n",
    "        <th>Revenue Share (%)</th>\n",
    "      </tr>\n",
    "    </thead>\n",
    "    <tbody>\n",
    "      {% for row in rows %}\n",
    "      <tr>\n",
    "        <td>{{ loop.index }}</td>\n",
    "        <td>{{ row.Product }}</td>\n",
    "        <td>{{ \"{:,}\".format(row.Total_Units) }}</td>\n",
    "        <td>{{ \"{:,.2f}\".format(row.Total_Revenue) }}</td>\n",
    "        <td>{{ \"{:,.2f}\".format(row.Avg_Price) }}</td>\n",
    "        <td>{{ \"{:.2f}\".format(row.Revenue_Share_Pct) }}%</td>\n",
    "      </tr>\n",
    "      {% endfor %}\n",
    "    </tbody>\n",
    "  </table>\n",
    "\n",
    "  <div class=\"total-box\">\n",
    "    💰 Grand Total Revenue: Rs {{ \"{:,.2f}\".format(total_revenue) }}\n",
    "  </div>\n",
    "\n",
    "  <h2>Monthly Revenue Summary</h2>\n",
    "  <table>\n",
    "    <thead>\n",
    "      <tr><th>Month</th><th>Revenue (Rs)</th></tr>\n",
    "    </thead>\n",
    "    <tbody>\n",
    "      {% for m in monthly %}\n",
    "      <tr>\n",
    "        <td>{{ m.Month }}</td>\n",
    "        <td>{{ \"{:,.2f}\".format(m.Monthly_Revenue) }}</td>\n",
    "      </tr>\n",
    "      {% endfor %}\n",
    "    </tbody>\n",
    "  </table>\n",
    "\n",
    "  <footer>Auto-generated by Sales Report Pipeline &copy; {{ year }}</footer>\n",
    "</body>\n",
    "</html>\n",
    "\"\"\"\n",
    "\n",
    "\n",
    "def generate_html(product_summary: pd.DataFrame,\n",
    "                  monthly_revenue: pd.DataFrame,\n",
    "                  path: str) -> None:\n",
    "    \"\"\"Render and save an HTML report using a Jinja2 template.\"\"\"\n",
    "\n",
    "    template = Template(HTML_TEMPLATE)\n",
    "\n",
    "    html_content = template.render(\n",
    "        report_date   = datetime.now().strftime(\"%d %B %Y, %H:%M\"),\n",
    "        year          = datetime.now().year,\n",
    "        rows          = product_summary.to_dict(orient=\"records\"),\n",
    "        monthly       = monthly_revenue.to_dict(orient=\"records\"),\n",
    "        total_revenue = product_summary[\"Total_Revenue\"].sum(),\n",
    "    )\n",
    "\n",
    "    with open(path, \"w\", encoding=\"utf-8\") as f:\n",
    "        f.write(html_content)\n",
    "\n",
    "    print(f\"[HTML] Saved → {path}\")\n",
    "\n",
    "\n",
    "generate_html(product_summary, monthly_revenue, HTML_PATH)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 6: GENERATE – Bar Chart of Monthly Revenue (Matplotlib)\n",
    "\n",
    "We create a bar chart that will be **embedded inside the PDF report**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[Chart] Saved → reports\\monthly_revenue_chart_20260312.png\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "C:\\Users\\shahi\\AppData\\Local\\Temp\\ipykernel_7428\\94432565.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown\n",
      "  plt.show()\n"
     ]
    }
   ],
   "source": [
    "def generate_bar_chart(monthly_revenue: pd.DataFrame, chart_path: str) -> None:\n",
    "    \"\"\"Save a bar chart of monthly revenue as a PNG image.\"\"\"\n",
    "\n",
    "    fig, ax = plt.subplots(figsize=(9, 4.5))\n",
    "\n",
    "    months = monthly_revenue[\"Month\"].astype(str).tolist()\n",
    "    values = monthly_revenue[\"Monthly_Revenue\"].tolist()\n",
    "    colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(months)))\n",
    "\n",
    "    bars = ax.bar(months, values, color=colors, edgecolor=\"#1F4E79\", linewidth=0.8)\n",
    "\n",
    "    # Value labels on top of each bar\n",
    "    for bar, val in zip(bars, values):\n",
    "        ax.text(\n",
    "            bar.get_x() + bar.get_width() / 2,\n",
    "            bar.get_height() + max(values) * 0.01,\n",
    "            f\"Rs {val/1e6:.1f}M\",\n",
    "            ha=\"center\", va=\"bottom\", fontsize=9, color=\"#1F4E79\", fontweight=\"bold\"\n",
    "        )\n",
    "\n",
    "    ax.set_title(\"Monthly Revenue Overview\", fontsize=14, fontweight=\"bold\",\n",
    "                 color=\"#1F4E79\", pad=14)\n",
    "    ax.set_xlabel(\"Month\", fontsize=11)\n",
    "    ax.set_ylabel(\"Revenue (Rs)\", fontsize=11)\n",
    "    ax.yaxis.set_major_formatter(\n",
    "        plt.FuncFormatter(lambda x, _: f\"Rs {x/1e6:.1f}M\")\n",
    "    )\n",
    "    ax.spines[[\"top\", \"right\"]].set_visible(False)\n",
    "    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.4)\n",
    "\n",
    "    plt.tight_layout()\n",
    "    plt.savefig(chart_path, dpi=150, bbox_inches=\"tight\")\n",
    "    plt.show()\n",
    "    plt.close()\n",
    "    print(f\"[Chart] Saved → {chart_path}\")\n",
    "\n",
    "\n",
    "generate_bar_chart(monthly_revenue, CHART_PATH)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 7: GENERATE – PDF Report (FPDF2)\n",
    "\n",
    "The PDF includes:\n",
    "- A styled title page header\n",
    "- Product summary table with alternating row colors\n",
    "- The bar chart of monthly revenue embedded as an image"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[PDF]   Saved → reports\\sales_report_20260312.pdf\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "C:\\Users\\shahi\\AppData\\Local\\Temp\\ipykernel_7428\\664579163.py:29: DeprecationWarning: The parameter \"ln\" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.\n",
      "  pdf.cell(0, 8, \"Product Summary\", ln=True)\n",
      "C:\\Users\\shahi\\AppData\\Local\\Temp\\ipykernel_7428\\664579163.py:81: DeprecationWarning: The parameter \"ln\" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.\n",
      "  pdf.cell(0, 8, \"Monthly Revenue Bar Chart\", ln=True)\n"
     ]
    }
   ],
   "source": [
    "def generate_pdf(product_summary: pd.DataFrame,\n",
    "                 monthly_revenue: pd.DataFrame,\n",
    "                 chart_path: str,\n",
    "                 path: str) -> None:\n",
    "    \"\"\"Generate a styled PDF report with a table and embedded bar chart.\"\"\"\n",
    "\n",
    "    pdf = FPDF()\n",
    "    pdf.set_auto_page_break(auto=True, margin=15)\n",
    "    pdf.add_page()\n",
    "\n",
    "    # ── Header banner ──────────────────────────────────────────────────────\n",
    "    pdf.set_fill_color(31, 78, 121)          # dark blue\n",
    "    pdf.rect(0, 0, 210, 28, style=\"F\")\n",
    "    pdf.set_text_color(255, 255, 255)\n",
    "    pdf.set_font(\"Helvetica\", \"B\", 18)\n",
    "    pdf.set_xy(0, 7)\n",
    "    pdf.cell(210, 10, \"Automated Sales Report\", align=\"C\")\n",
    "\n",
    "    pdf.set_font(\"Helvetica\", \"\", 10)\n",
    "    pdf.set_xy(0, 19)\n",
    "    pdf.cell(210, 6,\n",
    "             f\"Generated: {datetime.now().strftime('%d %B %Y, %H:%M')}\",\n",
    "             align=\"C\")\n",
    "\n",
    "    # ── Section: Product Summary ───────────────────────────────────────────\n",
    "    pdf.set_text_color(31, 78, 121)\n",
    "    pdf.set_font(\"Helvetica\", \"B\", 13)\n",
    "    pdf.set_xy(10, 34)\n",
    "    pdf.cell(0, 8, \"Product Summary\", ln=True)\n",
    "\n",
    "    # Table header\n",
    "    col_widths = [38, 28, 40, 40, 34]\n",
    "    headers    = [\"Product\", \"Units Sold\", \"Revenue (Rs)\",\n",
    "                  \"Avg Price (Rs)\", \"Share (%)\"]\n",
    "\n",
    "    pdf.set_fill_color(31, 78, 121)\n",
    "    pdf.set_text_color(255, 255, 255)\n",
    "    pdf.set_font(\"Helvetica\", \"B\", 9)\n",
    "    pdf.set_x(10)\n",
    "    for header, width in zip(headers, col_widths):\n",
    "        pdf.cell(width, 8, header, border=1, align=\"C\", fill=True)\n",
    "    pdf.ln()\n",
    "\n",
    "    # Table rows with alternating colors\n",
    "    pdf.set_font(\"Helvetica\", \"\", 9)\n",
    "    for i, row in product_summary.iterrows():\n",
    "        if i % 2 == 0:\n",
    "            pdf.set_fill_color(214, 228, 240)   # light blue\n",
    "        else:\n",
    "            pdf.set_fill_color(255, 255, 255)   # white\n",
    "        pdf.set_text_color(50, 50, 50)\n",
    "        pdf.set_x(10)\n",
    "\n",
    "        row_data = [\n",
    "            row[\"Product\"],\n",
    "            f\"{int(row['Total_Units']):,}\",\n",
    "            f\"{row['Total_Revenue']:,.2f}\",\n",
    "            f\"{row['Avg_Price']:,.2f}\",\n",
    "            f\"{row['Revenue_Share_Pct']:.2f}%\",\n",
    "        ]\n",
    "        for val, width in zip(row_data, col_widths):\n",
    "            pdf.cell(width, 7, str(val), border=1, align=\"C\", fill=True)\n",
    "        pdf.ln()\n",
    "\n",
    "    # Total row\n",
    "    pdf.set_fill_color(31, 78, 121)\n",
    "    pdf.set_text_color(255, 255, 255)\n",
    "    pdf.set_font(\"Helvetica\", \"B\", 9)\n",
    "    pdf.set_x(10)\n",
    "    total_rev = product_summary[\"Total_Revenue\"].sum()\n",
    "    total_units = product_summary[\"Total_Units\"].sum()\n",
    "    totals = [\"TOTAL\", f\"{int(total_units):,}\",\n",
    "              f\"{total_rev:,.2f}\", \"-\", \"100.00%\"]\n",
    "    for val, width in zip(totals, col_widths):\n",
    "        pdf.cell(width, 7, val, border=1, align=\"C\", fill=True)\n",
    "    pdf.ln(10)\n",
    "\n",
    "    # ── Section: Monthly Revenue Bar Chart ────────────────────────────────\n",
    "    pdf.set_text_color(31, 78, 121)\n",
    "    pdf.set_font(\"Helvetica\", \"B\", 13)\n",
    "    pdf.cell(0, 8, \"Monthly Revenue Bar Chart\", ln=True)\n",
    "    pdf.ln(2)\n",
    "\n",
    "    if os.path.exists(chart_path):\n",
    "        # Centre the chart on the page\n",
    "        img_w, img_h = 170, 85\n",
    "        x_pos = (210 - img_w) / 2\n",
    "        pdf.image(chart_path, x=x_pos, y=pdf.get_y(), w=img_w, h=img_h)\n",
    "        pdf.ln(img_h + 6)\n",
    "    else:\n",
    "        pdf.set_font(\"Helvetica\", \"I\", 10)\n",
    "        pdf.set_text_color(150, 0, 0)\n",
    "        pdf.cell(0, 8, \"[Chart image not found]\", ln=True)\n",
    "\n",
    "    # ── Footer ─────────────────────────────────────────────────────────────\n",
    "    pdf.set_y(-14)\n",
    "    pdf.set_font(\"Helvetica\", \"I\", 8)\n",
    "    pdf.set_text_color(170, 170, 170)\n",
    "    pdf.cell(0, 6,\n",
    "             f\"Auto-generated by Sales Report Pipeline  |  Page {pdf.page_no()}\",\n",
    "             align=\"C\")\n",
    "\n",
    "    pdf.output(path)\n",
    "    print(f\"[PDF]   Saved → {path}\")\n",
    "\n",
    "\n",
    "generate_pdf(product_summary, monthly_revenue, CHART_PATH, PDF_PATH)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 8: DELIVER – Email Function (SMTP / Gmail)\n",
    "\n",
    "The **Deliver** phase sends the generated report to a recipient via email using Python's built-in `smtplib` with **TLS encryption**.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def send_report_email(\n",
    "    sender_email   : str,\n",
    "    app_password   : str,\n",
    "    recipient_email: str,\n",
    "    attachment_path: str,\n",
    "    smtp_host      : str = \"smtp.gmail.com\",\n",
    "    smtp_port      : int = 587,\n",
    ") -> None:\n",
    "    \"\"\"\n",
    "    Send a report file as an email attachment via SMTP (TLS).\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    sender_email    : Gmail address of the sender.\n",
    "    app_password    : Gmail App Password (16-char string).\n",
    "    recipient_email : Destination email address.\n",
    "    attachment_path : Local path to the report file to attach.\n",
    "    smtp_host       : SMTP server (default: Gmail).\n",
    "    smtp_port       : SMTP port (587 = STARTTLS).\n",
    "    \"\"\"\n",
    "    if not os.path.exists(attachment_path):\n",
    "        print(f\"[Email] ERROR: File not found – {attachment_path}\")\n",
    "        return\n",
    "\n",
    "    # Build message\n",
    "    msg = MIMEMultipart()\n",
    "    msg[\"From\"]    = sender_email\n",
    "    msg[\"To\"]      = recipient_email\n",
    "    msg[\"Subject\"] = f\"Sales Report – {datetime.now().strftime('%d %B %Y')}\"\n",
    "\n",
    "    body = (\n",
    "        \"Hello,\\n\\n\"\n",
    "        \"Please find attached the automated sales report generated today.\\n\\n\"\n",
    "        \"This email was sent automatically by the Sales Report Pipeline.\\n\\n\"\n",
    "        \"Regards,\\nSales Analytics Team\"\n",
    "    )\n",
    "    msg.attach(MIMEText(body, \"plain\"))\n",
    "\n",
    "    # Attach the report file\n",
    "    with open(attachment_path, \"rb\") as f:\n",
    "        part = MIMEBase(\"application\", \"octet-stream\")\n",
    "        part.set_payload(f.read())\n",
    "\n",
    "    encoders.encode_base64(part)\n",
    "    part.add_header(\n",
    "        \"Content-Disposition\",\n",
    "        f'attachment; filename=\"{os.path.basename(attachment_path)}\"',\n",
    "    )\n",
    "    msg.attach(part)\n",
    "\n",
    "    # Send via SMTP with STARTTLS\n",
    "    try:\n",
    "        with smtplib.SMTP(smtp_host, smtp_port) as server:\n",
    "            server.ehlo()\n",
    "            server.starttls()          # Encrypt the connection\n",
    "            server.ehlo()\n",
    "            server.login(sender_email, app_password)\n",
    "            server.sendmail(sender_email, recipient_email, msg.as_string())\n",
    "        print(f\"[Email] Report sent successfully to {recipient_email}\")\n",
    "    except smtplib.SMTPAuthenticationError:\n",
    "        print(\"[Email] Authentication failed. Check your App Password.\")\n",
    "    except smtplib.SMTPException as e:\n",
    "        print(f\"[Email] SMTP error: {e}\")\n",
    "\n",
    "\n",
    "# ── USAGE () ─────────────────\n",
    "# SENDER_EMAIL    = \"youremail@gmail.com\"\n",
    "# APP_PASSWORD    = \"xxxx xxxx xxxx xxxx\"   # Gmail App Password\n",
    "# RECIPIENT_EMAIL = \"recipient@example.com\"\n",
    "\n",
    "# send_report_email(\n",
    "#     sender_email    = SENDER_EMAIL,\n",
    "#     app_password    = APP_PASSWORD,\n",
    "#     recipient_email = RECIPIENT_EMAIL,\n",
    "#     attachment_path = PDF_PATH,\n",
    "# )"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 9: Verify Generated Reports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "==================================================\n",
      "  Report Generation Summary\n",
      "==================================================\n",
      "  Excel   : sales_report_20260312.xlsx                    ✅  5.9 KB\n",
      "  HTML    : sales_report_20260312.html                    ✅  3.2 KB\n",
      "  PDF     : sales_report_20260312.pdf                     ✅  48.6 KB\n",
      "==================================================\n",
      "\n",
      "All reports saved in: ./reports/\n"
     ]
    }
   ],
   "source": [
    "print(\"=\" * 50)\n",
    "print(\"  Report Generation Summary\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "for label, fpath in [(\"Excel\", EXCEL_PATH), (\"HTML\", HTML_PATH), (\"PDF\", PDF_PATH)]:\n",
    "    exists = os.path.exists(fpath)\n",
    "    size   = os.path.getsize(fpath) / 1024 if exists else 0\n",
    "    status = f\"✅  {size:.1f} KB\" if exists else \"❌  NOT FOUND\"\n",
    "    print(f\"  {label:<8}: {os.path.basename(fpath):<45} {status}\")\n",
    "\n",
    "print(\"=\" * 50)\n",
    "print(f\"\\nAll reports saved in: ./{REPORTS_DIR}/\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Step 10: Discussion Questions\n",
    "\n",
    "### Q1. Explain the role of each step in the automated report pipeline: Extract → Transform → Generate → Deliver\n",
    "\n",
    "| Phase | Role |\n",
    "|-------|------|\n",
    "| **Extract** | Loads raw data from source systems (CSV, database, API) into memory. No changes are made — data is read as-is. This ensures a clean separation between data retrieval and processing. |\n",
    "| **Transform** | Cleans, enriches, and reshapes the data. Here we calculate `Revenue = Units_Sold × Unit_Price`, aggregate to product-level summaries, compute revenue share percentages, and build monthly totals. This is where business logic lives. |\n",
    "| **Generate** | Converts the transformed data into human-readable output formats — Excel (structured workbook), HTML (web page via Jinja2), and PDF (printable document via FPDF2 with chart). Each format serves a different audience or use case. |\n",
    "| **Deliver** | Distributes the finished reports to stakeholders automatically via email (SMTP/STARTTLS). This removes the need for manual file sharing and ensures timely delivery. |\n",
    "\n",
    "---\n",
    "\n",
    "### Q2. How does the pipeline ensure that Excel, HTML, and PDF reports are consistent in content?\n",
    "\n",
    "All three reports are generated **from the same DataFrames** — `product_summary` and `monthly_revenue` — which are computed once in the Transform phase. Because every format function receives these same objects as arguments, the figures, totals, and row order are identical across all outputs. There is a single source of truth; changing the data once updates all formats simultaneously.\n",
    "\n",
    "---\n",
    "\n",
    "### Q3. Modify the pipeline to include a bar chart of monthly revenue in the PDF report. *(Implemented in Step 6 & 7 above)*\n",
    "\n",
    "The modification involved two additions:\n",
    "1. **`generate_bar_chart()`** – Uses `matplotlib` to draw a styled bar chart of monthly revenue and saves it as a PNG file inside the `reports/` folder.\n",
    "2. **`generate_pdf()` updated** – After the product summary table, the function embeds the saved PNG into the PDF using `pdf.image()`, centering it on the page with appropriate dimensions.\n",
    "\n",
    "---\n",
    "\n",
    "### Q4. How would you secure email credentials when deploying this pipeline in production?\n",
    "\n",
    "Hardcoding credentials in source code is a serious security risk. In production, the following strategies should be used:\n",
    "\n",
    "1. **Environment Variables** – Store `SENDER_EMAIL` and `APP_PASSWORD` in OS environment variables and read them with `os.environ.get(\"APP_PASSWORD\")`. This keeps secrets out of the codebase.\n",
    "2. **`.env` file + `python-dotenv`** – Store credentials in a `.env` file, add it to `.gitignore`, and load it at runtime using the `python-dotenv` library.\n",
    "3. **Secrets Managers** – On cloud platforms, use services such as AWS Secrets Manager, Azure Key Vault, or HashiCorp Vault to store and rotate credentials securely.\n",
    "4. **CI/CD Secret Variables** – In automated pipelines (GitHub Actions, GitLab CI), inject secrets as masked environment variables so they never appear in logs.\n",
    "5. **OAuth2 instead of App Passwords** – For Gmail in production, prefer OAuth2 authentication over static App Passwords, as tokens can be scoped and revoked without changing the underlying account password."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Sender email loaded from env: ❌ Not set\n",
      "App password loaded from env: ❌ Not set\n"
     ]
    }
   ],
   "source": [
    "# Demonstration: secure credential loading using environment variables\n",
    "import os\n",
    "\n",
    "\n",
    "SENDER_EMAIL    = os.environ.get(\"REPORT_SENDER_EMAIL\", \"NOT_SET\")\n",
    "APP_PASSWORD    = os.environ.get(\"REPORT_APP_PASSWORD\", \"NOT_SET\")\n",
    "RECIPIENT_EMAIL = os.environ.get(\"REPORT_RECIPIENT\",   \"NOT_SET\")\n",
    "\n",
    "print(f\"Sender email loaded from env: {'✅' if SENDER_EMAIL != 'NOT_SET' else '❌ Not set'}\")\n",
    "print(f\"App password loaded from env: {'✅' if APP_PASSWORD != 'NOT_SET' else '❌ Not set'}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "venv",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.1"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}